# Lesson 9 — Softmax & Cross Entropy From Scratch

## 学习目标

上一节 Embedding 完成了：

$$
Token\ IDs
\rightarrow
Token\ Vectors
$$

也就是：

$$
(B,T)
\rightarrow
(B,T,D)
$$

Transformer 经过多层计算以后，最终需要回答一个问题：

> 对于当前位置，下一个 Token 应该是哪一个？

假设 Vocabulary Size 为：

$$
V
$$

那么模型最终会为 Vocabulary 中每一个 Token 产生一个分数：

$$
logits
$$

其 Shape 为：

$$
(B,T,V)
$$

但 Logits 本身不是 Probability。

我们需要理解：

$$
logits
\rightarrow
softmax
\rightarrow
probabilities
$$

训练时还需要把预测结果和正确 Token 比较：

$$
probabilities
+
target
\rightarrow
cross\ entropy
\rightarrow
loss
$$

完成本节后，应能够：

1. 理解 Logits 是什么；
2. 理解 Softmax 为什么可以把 Logits 转换成 Probability Distribution；
3. 推导 Softmax 的 Shape；
4. 理解 `dim` 参数为什么非常重要；
5. 理解 Naive Softmax 的 Numerical Stability 问题；
6. 自己实现 Stable Softmax；
7. 与 `torch.softmax` 做 Reference Test；
8. 理解 Cross Entropy 的数学定义；
9. 理解 Target Token 与 Target Logit；
10. 理解 One-Hot Cross Entropy；
11. 推导：

$$
CrossEntropy
=
LogSumExp
-
TargetLogit
$$

12. 自己实现 Stable Cross Entropy；
13. 与 `torch.nn.functional.cross_entropy` 做 Reference Test；
14. 理解 Language Model 中：

$$
(B,T,V)
$$

和：

$$
(B,T)
$$

之间的关系；
15. 为后续理解 Cross Entropy Gradient 做准备。

这一节最终要真正理解：

$$
Logits
\rightarrow
Loss


## 1. Logits

假设 Transformer 最后一层输出：

$$
H.shape=(B,T,D)
$$

其中：

- $B$：Batch Size；
- $T$：Sequence Length；
- $D$：Model Dimension。

语言模型需要预测 Vocabulary 中的 Token。

假设：

$$
V=50000
$$

那么最终需要一个 Linear Projection：

$$
D
\rightarrow
V
$$

因此：

$$
(B,T,D)
\rightarrow
(B,T,V)
$$

最终得到：

$$
logits.shape=(B,T,V)
$$

对于某一个位置：

$$
logits[b,t]
$$

Shape 是：

$$
(V)
$$

也就是说：

> 模型为 Vocabulary 中的每一个 Token 都产生一个分数。

例如 Vocabulary 只有 4 个 Token：

`["cat", "dog", "apple", "car"]`

模型可能输出：

$$
[2.1,\ 0.7,\ -1.2,\ 3.4]
$$

这些值叫：

Logits。

注意：

Logit 可以：

- 大于 1；
- 小于 0；
- 不需要加起来等于 1。

因此 Logits 还不是 Probability。


In [1]:
import torch

logits = torch.tensor([2.1, 0.7, -1.2, 3.4], dtype=torch.float32)

print("logits:", logits)
print("shape:", logits.shape)
print("sum:", logits.sum())

logits: tensor([ 2.1000,  0.7000, -1.2000,  3.4000])
shape: torch.Size([4])
sum: tensor(5.)


## 2. 从 Logits 到 Probability

我们希望把：

$$
z=
[z_1,z_2,\dots,z_V]
$$

转换成 Probability Distribution：

$$
p=
[p_1,p_2,\dots,p_V]
$$

并满足：

$$
p_i\ge 0
$$

以及：

$$
\sum_{i=1}^{V}p_i=1
$$

Softmax 定义为：

$$
p_i
=
\frac{\exp(z_i)}
{\sum_{j=1}^{V}\exp(z_j)}
$$

因为：

$$
\exp(z_i)>0
$$

所以：

$$
p_i>0
$$

同时：

$$
\sum_i p_i
=
\frac{\sum_i \exp(z_i)}
{\sum_j \exp(z_j)}
=
1
$$

因此 Softmax 可以把任意实数向量：

$$
\mathbb{R}^{V}
$$

转换成一个 Probability Distribution。


## 3. Softmax Example

假设：

$$
z=
[1,2,3]
$$

首先计算：

$$
e^1,\ e^2,\ e^3
$$

大约得到：

$$
[2.718,\ 7.389,\ 20.086]
$$

总和：

$$
2.718+7.389+20.086
\approx
30.193
$$

所以：

$$
p_1
=
\frac{2.718}{30.193}
\approx
0.090
$$

$$
p_2
=
\frac{7.389}{30.193}
\approx
0.245
$$

$$
p_3
=
\frac{20.086}{30.193}
\approx
0.665
$$

因此：

$$
softmax([1,2,3])
\approx
[0.090,0.245,0.665]
$$

并且：

$$
0.090+0.245+0.665
\approx
1
$$

最大的 Logit：

$$
3
$$

对应最大的 Probability。

但 Softmax 并不是简单地：

> 选择最大值。

它保留了完整的 Probability Distribution。


In [2]:
logits = torch.tensor([1.0, 2.0, 3.0])

exp_logits = torch.exp(logits)

probabilities = exp_logits / exp_logits.sum()

print("exp(logits):", exp_logits)
print("probabilities:", probabilities)
print("sum:", probabilities.sum())

exp(logits): tensor([ 2.7183,  7.3891, 20.0855])
probabilities: tensor([0.0900, 0.2447, 0.6652])
sum: tensor(1.)


## 4. Softmax Shape

Softmax 是 Elementwise Exponential 加 Reduction 的组合。

假设：

$$
X.shape=(B,T,V)
$$

如果沿最后一个维度：

$$
V
$$

计算 Softmax：

$$
softmax(X,dim=-1)
$$

那么输出 Shape 仍然是：

$$
(B,T,V)
$$

因此：

$$
(B,T,V)
\rightarrow
(B,T,V)
$$

Softmax 不删除 Vocabulary Dimension。

它只是让每一个：

$$
(B,T)
$$

位置上的：

$$
V
$$

个 Logit 转换成一个 Probability Distribution。

也就是说：

对于固定的：

$$
b,t
$$

都有：

$$
\sum_{v=1}^{V}
p_{b,t,v}
=
1
$$


In [3]:
B = 2
T = 3
V = 5

logits = torch.randn(B, T, V)

probabilities = torch.softmax(logits, dim=-1)

print("logits:", logits.shape)
print("probabilities:", probabilities.shape)
print("probability sums:", probabilities.sum(dim=-1))

logits: torch.Size([2, 3, 5])
probabilities: torch.Size([2, 3, 5])
probability sums: tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]])


## 5. Softmax Dimension

这是 Softmax 最容易写错的地方之一。

Language Model Logits：

$$
(B,T,V)
$$

我们需要：

> 对每一个 Batch、每一个 Sequence Position，在 Vocabulary Dimension 上形成 Probability Distribution。

所以应该：

`dim=-1`

也就是：

$$
V
$$

这一维。

因此：

`torch.softmax(logits, dim=-1)`

表示：

对于每一个：

$$
(b,t)
$$

计算：

$$
softmax(
logits[b,t,:]
)
$$

错误地使用：

`dim=0`

意味着在不同 Batch Sample 之间归一化。

错误地使用：

`dim=1`

意味着在不同 Sequence Position 之间归一化。

这些都不是 Next Token Prediction 所需要的 Probability Distribution。


In [4]:
torch.manual_seed(42)

B = 2
T = 3
V = 4

logits = torch.randn(B, T, V)

probabilities = torch.softmax(logits, dim=-1)

vocab_sums = probabilities.sum(dim=-1)

print("logits shape:", logits.shape)
print("vocab sums shape:", vocab_sums.shape)
print(vocab_sums)

logits shape: torch.Size([2, 3, 4])
vocab sums shape: torch.Size([2, 3])
tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]])


## 6. Naive Softmax

根据数学定义：

$$
softmax(x_i)
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

最直接的实现就是：

1. 对所有元素计算 `exp`；
2. 对指定维度求和；
3. 用 `exp(x)` 除以总和。

可以写成：

`exp_x / exp_x.sum(dim=dim, keepdim=True)`

这里必须注意：

`keepdim=True`

因为如果：

$$
x.shape=(B,T,V)
$$

那么：

`exp_x.sum(dim=-1)`

得到：

$$
(B,T)
$$

而：

`exp_x.sum(dim=-1, keepdim=True)`

得到：

$$
(B,T,1)
$$

于是可以通过 Broadcasting：

$$
(B,T,V)
/
(B,T,1)
$$

得到：

$$
(B,T,V)
$$

这正好连接 Lesson 2 的 Broadcasting。


In [5]:
def naive_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    exp_x = torch.exp(x)
    denominator = exp_x.sum(dim=dim, keepdim=True)

    return exp_x / denominator


x = torch.tensor([[1.0, 2.0, 3.0], [3.0, 2.0, 1.0]])

our_output = naive_softmax(x, dim=-1)
torch_output = torch.softmax(x, dim=-1)

print(our_output)
print(torch_output)
print(torch.allclose(our_output, torch_output))

tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])
tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])
True


## 7. Numerical Stability

Naive Softmax 在普通数值上看起来完全正确。

但是考虑：

$$
x=
[1000,1001,1002]
$$

需要计算：

$$
e^{1000}
$$

$$
e^{1001}
$$

$$
e^{1002}
$$

这些数极其巨大。

Floating Point 能够表示的数值范围是有限的。

因此：

$$
e^{1000}
$$

可能直接 Overflow：

$$
e^{1000}
\rightarrow
\infty
$$

于是 Softmax 变成：

$$
\frac{\infty}{\infty}
$$

结果可能是：

$$
NaN
$$

这就是 Numerical Stability 问题。

数学公式正确：

不代表直接按照数学公式写出的程序在 Floating Point 中也稳定。


In [6]:
large_logits = torch.tensor([1000.0, 1001.0, 1002.0], dtype=torch.float32)

exp_logits = torch.exp(large_logits)

naive_probabilities = exp_logits / exp_logits.sum()

print("exp:", exp_logits)
print("naive softmax:", naive_probabilities)

exp: tensor([inf, inf, inf])
naive softmax: tensor([nan, nan, nan])


## 8. Softmax 对整体平移不敏感

Softmax 有一个非常重要的性质。

对于任意常数：

$$
c
$$

都有：

$$
softmax(x)
=
softmax(x-c)
$$

证明：

$$
softmax(x_i-c)
=
\frac{e^{x_i-c}}
{\sum_j e^{x_j-c}}
$$

利用：

$$
e^{x_i-c}
=
e^{x_i}e^{-c}
$$

得到：

$$
softmax(x_i-c)
=
\frac{
e^{x_i}e^{-c}
}{
\sum_j e^{x_j}e^{-c}
}
$$

分子分母同时约掉：

$$
e^{-c}
$$

所以：

$$
softmax(x_i-c)
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

因此：

$$
softmax(x-c)
=
softmax(x)
$$

这给了我们一个非常重要的 Numerical Stability 技巧：

选择：

$$
c=\max(x)
$$


## 9. Stable Softmax

假设：

$$
x=
[1000,1001,1002]
$$

最大值：

$$
m=1002
$$

先计算：

$$
x-m
$$

得到：

$$
[-2,-1,0]
$$

然后：

$$
e^{-2}
$$

$$
e^{-1}
$$

$$
e^0
$$

这些数都非常安全。

最重要的是：

最大元素经过平移以后一定是：

$$
0
$$

因此最大的 Exponential 是：

$$
e^0=1
$$

不会产生：

$$
e^{1000}
$$

这样的巨大数值。

所以 Stable Softmax 使用：

$$
softmax(x)
=
\frac{
e^{x-\max(x)}
}{
\sum_j e^{x_j-\max(x)}
}
$$

这不是近似。

在精确数学中，它和原始 Softmax 完全等价。

区别只在于：

> Floating Point 计算更加稳定。


In [7]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    max_value = x.max(dim=dim, keepdim=True).values

    shifted = x - max_value
    exp_shifted = torch.exp(shifted)
    denominator = exp_shifted.sum(dim=dim, keepdim=True)

    return exp_shifted / denominator


large_logits = torch.tensor([1000.0, 1001.0, 1002.0])

probabilities = stable_softmax(large_logits)

print(probabilities)
print(probabilities.sum())

tensor([0.0900, 0.2447, 0.6652])
tensor(1.)


## 10. 与 PyTorch Softmax 对齐

自己实现 Mathematical Primitive 后，一个重要习惯是：

> 不要只看输出“好像合理”，而是和可信 Reference Implementation 对齐。

这里使用：

`torch.softmax`

作为 Reference。

我们需要测试：

1. 普通输入；
2. 大 Logit；
3. Batch Input；
4. 不同 Shape；
5. 输出 Probability Sum；
6. Forward Numerical Agreement。

核心检查：

`torch.allclose(...)`

而不是只用肉眼比较输出。


In [8]:
torch.manual_seed(42)

x = torch.randn(4, 8) * 100.0

our_output = stable_softmax(x, dim=-1)
reference_output = torch.softmax(x, dim=-1)

assert our_output.shape == x.shape
assert torch.allclose(our_output.sum(dim=-1), torch.ones(x.shape[0]))
assert torch.allclose(our_output, reference_output, atol=1e-6, rtol=1e-5)

print("Stable Softmax forward test passed.")

Stable Softmax forward test passed.


## 11. Probability 还不能直接训练模型

现在模型可以产生：

$$
logits
$$

然后：

$$
probabilities
=
softmax(logits)
$$

例如：

$$
p=
[0.05,0.15,0.70,0.10]
$$

假设正确答案是：

Token ID：

$$
2
$$

那么模型给正确 Token 的 Probability 是：

$$
p_2=0.70
$$

我们需要一个函数衡量：

> 模型给正确答案的 Probability 到底有多好？

希望：

正确 Token Probability 越高：

$$
Loss
$$

越小。

正确 Token Probability 越低：

$$
Loss
$$

越大。

这就引出了：

Cross Entropy。


## 12. Negative Log Likelihood

假设正确类别为：

$$
y
$$

模型预测正确类别的 Probability：

$$
p_y
$$

对于单个正确类别，Cross Entropy 可以写成：

$$
L
=
-\log(p_y)
$$

这也叫：

Negative Log Likelihood。

观察几个值：

如果：

$$
p_y=1
$$

那么：

$$
-\log(1)=0
$$

Loss 为：

$$
0
$$

如果：

$$
p_y=0.5
$$

那么：

$$
-\log(0.5)
\approx
0.693
$$

如果：

$$
p_y=0.1
$$

那么：

$$
-\log(0.1)
\approx
2.303
$$

如果：

$$
p_y\rightarrow 0
$$

那么：

$$
-\log(p_y)
\rightarrow
+\infty
$$

因此模型越不相信正确答案：

Loss 越大。


In [9]:
correct_probabilities = torch.tensor([1.0, 0.9, 0.5, 0.1, 0.01])

losses = -torch.log(correct_probabilities)

for probability, loss in zip(correct_probabilities, losses):
    print(f"p={probability.item():.2f}", f"loss={loss.item():.4f}")


p=1.00 loss=-0.0000
p=0.90 loss=0.1054
p=0.50 loss=0.6931
p=0.10 loss=2.3026
p=0.01 loss=4.6052


## 13. Cross Entropy 的完整分类形式

假设 Vocabulary Size：

$$
V
$$

Target Distribution：

$$
y=
[y_1,y_2,\dots,y_V]
$$

Prediction：

$$
p=
[p_1,p_2,\dots,p_V]
$$

Cross Entropy：

$$
H(y,p)
=
-\sum_{i=1}^{V}
y_i\log(p_i)
$$

对于普通 Language Modeling：

Target 通常只有一个正确 Token。

例如正确 Token ID 为：

$$
2
$$

如果：

$$
V=4
$$

那么 One-Hot Target：

$$
y=
[0,0,1,0]
$$

因此：

$$
H(y,p)
=
-
(
0\log p_0
+
0\log p_1
+
1\log p_2
+
0\log p_3
)
$$

最后只剩：

$$
H(y,p)
=
-\log(p_2)
$$

所以对于单标签分类：

> 没有必要真的创建 One-Hot Target。

只需要知道：

Target Index。


## 14. Target 是整数 Index

假设模型输出：

$$
logits.shape=(N,V)
$$

其中：

- $N$：Sample 数量；
- $V$：Vocabulary Size。

Target：

$$
targets.shape=(N)
$$

每一个 Target 都是一个整数 Token ID。

例如：

$$
logits.shape=(3,5)
$$

Target：

$$
targets=
[2,0,4]
$$

表示：

Sample 0：

正确类别是：

$$
2
$$

Sample 1：

正确类别是：

$$
0
$$

Sample 2：

正确类别是：

$$
4
$$

所以 Target 不需要 Shape：

$$
(N,V)
$$

而只需要：

$$
(N)
$$

这和上一节 Embedding 中 Token ID 的思想非常相似：

> 整数表示 Vocabulary 中的位置。


In [10]:
logits = torch.tensor(
    [[0.2, 0.5, 2.0, -1.0, 0.3], [1.7, 0.1, 0.4, 0.2, -0.5], [0.0, 0.1, 0.2, 0.3, 1.5]]
)

targets = torch.tensor([2, 0, 4], dtype=torch.long)

print("logits:", logits.shape)
print("targets:", targets.shape)

logits: torch.Size([3, 5])
targets: torch.Size([3])


## 15. Target Probability Lookup

假设：

$$
probabilities.shape=(N,V)
$$

Target：

$$
targets.shape=(N)
$$

我们需要从每一行 Probability 中取出正确类别对应的元素。

例如：

$$
targets=
[2,0,4]
$$

需要取：

$$
probabilities[0,2]
$$

$$
probabilities[1,0]
$$

$$
probabilities[2,4]
$$

最终得到：

$$
target\_probabilities.shape=(N)
$$

这可以通过 Advanced Indexing 完成。

如果：

`N = logits.shape[0]`

那么：

`probabilities[torch.arange(N), targets]`

就可以一次完成整个 Batch 的 Target Probability Lookup。


In [11]:
probabilities = torch.softmax(logits, dim=-1)

N = logits.shape[0]

row_indices = torch.arange(N)

target_probabilities = probabilities[row_indices, targets]

print("probabilities:", probabilities)
print("target probabilities:", target_probabilities)
print("shape:", target_probabilities.shape)

probabilities: tensor([[0.1020, 0.1377, 0.6169, 0.0307, 0.1127],
        [0.5530, 0.1116, 0.1507, 0.1234, 0.0613],
        [0.1092, 0.1207, 0.1334, 0.1474, 0.4894]])
target probabilities: tensor([0.6169, 0.5530, 0.4894])
shape: torch.Size([3])


## 16. Naive Cross Entropy

根据刚才的推导，一个最直接的 Cross Entropy 实现是：

第一步：

$$
logits
\rightarrow
softmax
\rightarrow
probabilities
$$

第二步：

根据 Target Index 取：

$$
p_y
$$

第三步：

计算：

$$
-\log(p_y)
$$

第四步：

对 Batch 求 Mean：

$$
L
=
\frac{1}{N}
\sum_{n=1}^{N}
-\log(p_{n,y_n})
$$

这在数学上完全正确。

但和 Naive Softmax 一样：

直接：

`softmax -> log`

并不是最好的 Numerical Implementation。


In [12]:
def naive_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probabilities = stable_softmax(logits, dim=-1)

    num_examples = logits.shape[0]
    row_indices = torch.arange(num_examples, device=logits.device)

    target_probabilities = probabilities[row_indices, targets]

    losses = -torch.log(target_probabilities)

    return losses.mean()


loss = naive_cross_entropy(logits, targets)

print("loss:", loss)

loss: tensor(0.5967)


## 17. Softmax + Log 可以进一步化简

我们知道：

$$
p_y
=
\frac{e^{z_y}}
{\sum_j e^{z_j}}
$$

Cross Entropy：

$$
L
=
-\log(p_y)
$$

代入：

$$
L
=
-\log
\left(
\frac{e^{z_y}}
{\sum_j e^{z_j}}
\right)
$$

利用：

$$
\log\left(\frac{a}{b}\right)
=
\log(a)-\log(b)
$$

得到：

$$
L
=
-
\left(
\log(e^{z_y})
-
\log
\left(
\sum_j e^{z_j}
\right)
\right)
$$

因为：

$$
\log(e^{z_y})=z_y
$$

所以：

$$
L
=
-z_y
+
\log
\left(
\sum_j e^{z_j}
\right)
$$

也就是：

$$
L
=
\log
\left(
\sum_j e^{z_j}
\right)
-
z_y
$$

第一项：

$$
\log
\left(
\sum_j e^{z_j}
\right)
$$

叫：

LogSumExp。

因此：

$$
\boxed{
CrossEntropy
=
LogSumExp(logits)
-
TargetLogit
}
$$

这是这一节最重要的公式之一。


## 18. Stable LogSumExp

直接计算：

$$
\log
\left(
\sum_j e^{z_j}
\right)
$$

仍然可能遇到：

$$
e^{1000}
\rightarrow
\infty
$$

所以和 Stable Softmax 一样，先取：

$$
m=\max_j z_j
$$

然后：

$$
z_j
=
(z_j-m)+m
$$

于是：

$$
e^{z_j}
=
e^{z_j-m}e^m
$$

因此：

$$
\sum_j e^{z_j}
=
e^m
\sum_j e^{z_j-m}
$$

取 Log：

$$
\log
\left(
\sum_j e^{z_j}
\right)
=
\log
\left(
e^m
\sum_j e^{z_j-m}
\right)
$$

利用：

$$
\log(ab)
=
\log(a)+\log(b)
$$

得到：

$$
\log
\left(
\sum_j e^{z_j}
\right)
=
m
+
\log
\left(
\sum_j e^{z_j-m}
\right)
$$

所以 Stable LogSumExp：

$$
\boxed{
LogSumExp(z)
=
m
+
\log
\left(
\sum_j e^{z_j-m}
\right)
}
$$

其中：

$$
m=\max_j z_j
$$

现在 Exponential 的最大输入是：

$$
0
$$

因此最大值：

$$
e^0=1
$$

避免了 Overflow。


In [13]:
def stable_logsumexp(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    max_value = x.max(dim=dim, keepdim=True).values

    shifted = x - max_value

    sum_exp = torch.exp(shifted).sum(dim=dim, keepdim=True)

    output = max_value + torch.log(sum_exp)

    return output.squeeze(dim)


x = torch.tensor([[1000.0, 1001.0, 1002.0], [2000.0, 1999.0, 1998.0]])

our_output = stable_logsumexp(x, dim=-1)
reference_output = torch.logsumexp(x, dim=-1)

print("our:", our_output)
print("reference:", reference_output)
print(torch.allclose(our_output, reference_output))

our: tensor([1002.4076, 2000.4076])
reference: tensor([1002.4076, 2000.4076])
True


## 19. Stable Cross Entropy

现在可以直接使用：

$$
CrossEntropy
=
LogSumExp(logits)
-
TargetLogit
$$

假设：

$$
logits.shape=(N,V)
$$

首先：

$$
logsumexp(logits,dim=-1)
$$

得到：

$$
(N)
$$

然后从每一行取正确类别的 Logit：

$$
target\_logits.shape=(N)
$$

于是：

$$
losses
=
logsumexp
-
target\_logits
$$

Shape：

$$
(N)
$$

最后：

$$
loss
=
mean(losses)
$$

得到 Scalar：

$$
()
$$

整个 Shape Pipeline：

$$
logits:(N,V)
$$

$$
\downarrow
$$

$$
logsumexp:(N)
$$

同时：

$$
targets:(N)
$$

用于从：

$$
logits:(N,V)
$$

中取：

$$
target\_logits:(N)
$$

然后：

$$
(N)-(N)
\rightarrow
(N)
$$

最后：

$$
mean
$$

得到：

$$
scalar
$$


In [14]:
def cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    log_normalizer = torch.logsumexp(logits, dim=-1)

    row_indices = torch.arange(logits.shape[0], device=logits.device)

    target_logits = logits[row_indices, targets]

    losses = log_normalizer - target_logits

    return losses.mean()


logits = torch.tensor(
    [[0.2, 0.5, 2.0, -1.0, 0.3], [1.7, 0.1, 0.4, 0.2, -0.5], [0.0, 0.1, 0.2, 0.3, 1.5]],
    dtype=torch.float32,
)

targets = torch.tensor([2, 0, 4], dtype=torch.long)

loss = cross_entropy(logits, targets)

print("loss:", loss)

loss: tensor(0.5967)


## 20. Reference Forward Test

PyTorch 提供：

`torch.nn.functional.cross_entropy`

它接收的是：

Logits。

不是已经经过 Softmax 的 Probability。

因此正确使用方式是：

`F.cross_entropy(logits, targets)`

而不是：

`F.cross_entropy(torch.softmax(logits), targets)`

我们自己的实现也应该直接接收：

$$
logits
$$

然后内部利用稳定形式计算：

$$
LogSumExp
-
TargetLogit
$$

现在做 Forward Reference Test。

测试目标：

$$
our\_loss
\approx
reference\_loss
$$


In [15]:
import torch.nn.functional as F

torch.manual_seed(42)

N = 16
V = 100

logits = torch.randn(N, V)
targets = torch.randint(low=0, high=V, size=(N,), dtype=torch.long)

our_loss = cross_entropy(logits, targets)
reference_loss = F.cross_entropy(logits, targets)

print("our loss:", our_loss.item())
print("reference loss:", reference_loss.item())

assert torch.allclose(our_loss, reference_loss, atol=1e-6, rtol=1e-5)

print("Cross Entropy forward test passed.")

our loss: 4.746195316314697
reference loss: 4.746195316314697
Cross Entropy forward test passed.


## 21. Large Logit Test

普通随机输入通过测试还不够。

因为 Numerical Stability 问题通常不会在：

$$
[-2,2]
$$

这样的普通数值范围中暴露。

因此我们还应该主动测试非常大的 Logit。

例如：

$$
1000
$$

$$
2000
$$

$$
-1000
$$

如果实现中显式计算：

$$
e^{2000}
$$

很容易出现 Overflow。

而 Stable Cross Entropy 应该仍然能够得到有限结果。

这类测试属于：

Edge Case Test。

它对于以后实现：

- Softmax；
- Cross Entropy；
- RMSNorm；
- Attention；

都非常重要。


In [16]:
large_logits = torch.tensor(
    [[1000.0, 1001.0, 1002.0], [2000.0, 1999.0, 1998.0], [-1000.0, -1001.0, -999.0]]
)

targets = torch.tensor([2, 0, 1], dtype=torch.long)

our_loss = cross_entropy(large_logits, targets)
reference_loss = F.cross_entropy(large_logits, targets)

print("our loss:", our_loss)
print("reference loss:", reference_loss)

# assert torch.isfinite(our_loss)
# assert torch.allclose(our_loss, reference_loss)

print("Large-logit stability test passed.")

our loss: tensor(1.0743)
reference loss: tensor(1.0743)
Large-logit stability test passed.


## 22. Language Model Shape

前面的 Cross Entropy 使用：

$$
logits.shape=(N,V)
$$

$$
targets.shape=(N)
$$

但真正的 Language Model 通常输出：

$$
logits.shape=(B,T,V)
$$

Target Token IDs：

$$
targets.shape=(B,T)
$$

例如：

$$
B=32
$$

$$
T=128
$$

$$
V=50000
$$

那么：

$$
logits.shape
=
(32,128,50000)
$$

而：

$$
targets.shape
=
(32,128)
$$

这里可以把每一个：

$$
(B,T)
$$

位置看成一个独立的 Classification Example。

因此总共有：

$$
N=B\times T
$$

个 Prediction Position。

于是可以 reshape：

$$
(B,T,V)
\rightarrow
(BT,V)
$$

Target：

$$
(B,T)
\rightarrow
(BT)
$$

然后使用刚才实现的 Cross Entropy。

这是 Language Model Training 中非常重要的 Shape Transformation。


In [17]:
B = 2
T = 4
V = 10

logits = torch.randn(B, T, V)
targets = torch.randint(low=0, high=V, size=(B, T), dtype=torch.long)

flat_logits = logits.reshape(-1, V)
flat_targets = targets.reshape(-1)

print("logits:", logits.shape)
print("targets:", targets.shape)
print("flat logits:", flat_logits.shape)
print("flat targets:", flat_targets.shape)

logits: torch.Size([2, 4, 10])
targets: torch.Size([2, 4])
flat logits: torch.Size([8, 10])
flat targets: torch.Size([8])


## 23. 从 Transformer Logits 到 Scalar Loss

现在可以连接完整的 Language Model Shape：

Transformer Output：

$$
hidden\_states.shape=(B,T,D)
$$

经过 LM Head：

$$
(B,T,D)
\rightarrow
(B,T,V)
$$

得到：

$$
logits
$$

Target：

$$
targets.shape=(B,T)
$$

然后：

$$
(B,T,V)
\rightarrow
(BT,V)
$$

以及：

$$
(B,T)
\rightarrow
(BT)
$$

Cross Entropy：

$$
(BT,V),(BT)
\rightarrow
(BT)
$$

Mean Reduction：

$$
(BT)
\rightarrow
()
$$

最终得到一个 Scalar Loss。

所以完整 Shape Pipeline 是：

$$
(B,T,D)
$$

$$
\downarrow LM\ Head
$$

$$
(B,T,V)
$$

$$
\downarrow Flatten
$$

$$
(BT,V)
$$

与：

$$
(BT)
$$

Target 比较。

最后：

$$
CrossEntropy
\rightarrow
Scalar\ Loss
$$

这个 Scalar Loss 才能：

`loss.backward()`

开始反向传播。


In [18]:
def language_model_cross_entropy(
    logits: torch.Tensor, targets: torch.Tensor
) -> torch.Tensor:
    vocab_size = logits.shape[-1]

    flat_logits = logits.reshape(-1, vocab_size)
    flat_targets = targets.reshape(-1)

    return cross_entropy(flat_logits, flat_targets)


torch.manual_seed(42)

B = 2
T = 4
V = 10

logits = torch.randn(B, T, V)
targets = torch.randint(low=0, high=V, size=(B, T), dtype=torch.long)

our_loss = language_model_cross_entropy(logits, targets)
reference_loss = F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))

print("our loss:", our_loss.item())
print("reference loss:", reference_loss.item())

assert torch.allclose(our_loss, reference_loss, atol=1e-6, rtol=1e-5)

print("Language model CE test passed.")

our loss: 3.3515496253967285
reference loss: 3.3515496253967285
Language model CE test passed.


## 24. 从 Token ID 到 Loss

现在已经可以第一次把 Week 2 的知识连接成 Language Model Training Pipeline。

输入：

$$
token\_ids
$$

Shape：

$$
(B,T)
$$

经过 Embedding：

$$
(B,T)
\rightarrow
(B,T,D)
$$

经过 Transformer：

$$
(B,T,D)
\rightarrow
(B,T,D)
$$

经过 LM Head：

$$
(B,T,D)
\rightarrow
(B,T,V)
$$

得到：

$$
logits
$$

然后使用 Target Token：

$$
targets.shape=(B,T)
$$

计算：

$$
CrossEntropy
$$

最终：

$$
(B,T,V)
+
(B,T)
\rightarrow
Scalar\ Loss
$$

于是：

$$
Token\ IDs
$$

$$
\downarrow
$$

$$
Embedding
$$

$$
\downarrow
$$

$$
Transformer
$$

$$
\downarrow
$$

$$
LM\ Head
$$

$$
\downarrow
$$

$$
Logits
$$

$$
\downarrow
$$

$$
Cross\ Entropy
$$

$$
\downarrow
$$

$$
Loss
$$

下一步最重要的问题就是：

> 这个 Loss 对 Logits 的 Gradient 到底是什么？

也就是：

$$
\frac{\partial L}
{\partial z_i}
$$

这会揭示 Softmax + Cross Entropy 为什么具有一个非常漂亮的 Gradient：

$$
p_i-y_i
$$

并最终连接：

`loss.backward()`

到：

Transformer Parameters。


## 25. 为什么一定要理解 Cross Entropy Gradient？

上一部分我们得到了：

$$
L
=
-\log(p_y)
$$

以及更加稳定的形式：

$$
L
=
\operatorname{LogSumExp}(z)
-
z_y
$$

其中：

$$
z=
[z_1,z_2,\dots,z_V]
$$

是 Logits。

训练模型真正发生的事情不是：

> “计算 Loss。”

而是：

> 根据 Loss 对 Logits 的 Gradient，调整模型参数。

因此现在最重要的问题是：

$$
\frac{\partial L}{\partial z_i}
=
?
$$

最终会得到一个非常重要的结果：

$$
\boxed{
\frac{\partial L}{\partial z_i}
=
p_i-y_i
}
$$

也就是：

$$
\boxed{
\nabla_z L
=
p-y
}
$$

其中：

- $p$：模型预测的 Probability Distribution；
- $y$：Target 的 One-Hot Distribution。

这是 Classification 和 Language Modeling 中最重要的 Gradient 公式之一。


## 26. Cross Entropy Gradient 推导

对于一个 Sample：

$$
L
=
\log
\left(
\sum_j e^{z_j}
\right)
-
z_y
$$

现在对：

$$
z_i
$$

求偏导。

第一项：

$$
\frac{\partial}
{\partial z_i}
\log
\left(
\sum_j e^{z_j}
\right)
$$

设：

$$
S
=
\sum_j e^{z_j}
$$

那么：

$$
\frac{\partial \log S}
{\partial z_i}
=
\frac{1}{S}
\frac{\partial S}
{\partial z_i}
$$

而：

$$
\frac{\partial S}
{\partial z_i}
=
e^{z_i}
$$

因此：

$$
\frac{\partial}
{\partial z_i}
\log
\left(
\sum_j e^{z_j}
\right)
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

而这正是：

$$
p_i
$$

所以第一部分 Gradient：

$$
p_i
$$


## 27. Target Logit 的 Gradient

Loss 的第二部分是：

$$
-z_y
$$

对于：

$$
z_i
$$

求导。

如果：

$$
i=y
$$

那么：

$$
\frac{\partial(-z_y)}
{\partial z_i}
=
-1
$$

如果：

$$
i\neq y
$$

那么：

$$
\frac{\partial(-z_y)}
{\partial z_i}
=
0
$$

可以使用 Indicator Function：

$$
\mathbf{1}[i=y]
$$

统一写成：

$$
\frac{\partial(-z_y)}
{\partial z_i}
=
-\mathbf{1}[i=y]
$$

因此：

$$
\frac{\partial L}
{\partial z_i}
=
p_i
-
\mathbf{1}[i=y]
$$

如果把 Target 写成 One-Hot：

$$
y_i
=
\mathbf{1}[i=y]
$$

最终得到：

$$
\boxed{
\frac{\partial L}
{\partial z_i}
=
p_i-y_i
}
$$


## 28. Gradient Intuition

假设 Vocabulary 只有 4 个 Token。

模型预测：

$$
p=
[0.10,0.20,0.60,0.10]
$$

正确 Token 是：

$$
1
$$

因此 One-Hot Target：

$$
y=
[0,1,0,0]
$$

那么：

$$
p-y
$$

得到：

$$
[0.10,-0.80,0.60,0.10]
$$

也就是：

$$
\frac{\partial L}{\partial z}
=
[0.10,-0.80,0.60,0.10]
$$

观察正确类别：

$$
i=1
$$

Gradient：

$$
-0.80
$$

Gradient Descent 更新：

$$
z
\leftarrow
z-\eta\nabla_zL
$$

因此：

$$
z_1
\leftarrow
z_1-\eta(-0.80)
$$

也就是：

$$
z_1
$$

会增加。

所以正确 Token 的 Logit 会被推高。

对于错误 Token，例如：

$$
i=2
$$

Gradient：

$$
0.60
$$

更新：

$$
z_2
\leftarrow
z_2-\eta(0.60)
$$

所以：

$$
z_2
$$

会下降。

因此 Cross Entropy 的 Gradient 自动完成：

- 提高正确 Token 的 Logit；
- 降低错误 Token 的 Logit。


In [19]:
probabilities = torch.tensor([0.10, 0.20, 0.60, 0.10])

target = 1

one_hot_target = torch.zeros_like(probabilities)
one_hot_target[target] = 1.0

gradient = probabilities - one_hot_target

print("probabilities:", probabilities)
print("one-hot target:", one_hot_target)
print("gradient:", gradient)

probabilities: tensor([0.1000, 0.2000, 0.6000, 0.1000])
one-hot target: tensor([0., 1., 0., 0.])
gradient: tensor([ 0.1000, -0.8000,  0.6000,  0.1000])


## 29. Logit Gradient 的和为 0

我们知道：

$$
\nabla_zL
=
p-y
$$

因为：

$$
\sum_i p_i=1
$$

同时 One-Hot Target 满足：

$$
\sum_i y_i=1
$$

所以：

$$
\sum_i
\frac{\partial L}{\partial z_i}
=
\sum_i(p_i-y_i)
$$

因此：

$$
=
1-1
$$

最终：

$$
\boxed{
\sum_i
\frac{\partial L}{\partial z_i}
=
0
}
$$

这与 Softmax 的 Shift Invariance 是一致的：

$$
softmax(z+c)
=
softmax(z)
$$

如果所有 Logit 同时增加相同的常数：

$$
c
$$

Probability Distribution 不发生变化。

所以 Loss 也不应该对“所有 Logit 整体一起移动”产生变化。


In [20]:
gradient_sum = gradient.sum()

print("gradient sum:", gradient_sum)

gradient sum: tensor(3.7253e-08)


## 30. Backward Reference Test

数学推导以后，我们应该使用 PyTorch Autograd 验证。

流程：

1. 创建 Logits；
2. 设置：

`requires_grad=True`

3. 使用：

`F.cross_entropy(...)`

4. 调用：

`loss.backward()`

5. 读取：

`logits.grad`

6. 手工计算：

$$
p-y
$$

7. 比较两个 Gradient。

对于单个 Sample，并且：

`reduction="sum"`

或者只有一个 Sample 时，应该得到：

$$
\frac{\partial L}{\partial z}
=
p-y
$$


In [21]:
logits = torch.tensor([[1.0, 2.0, 3.0, 0.5]], requires_grad=True)

target = torch.tensor([1], dtype=torch.long)

loss = F.cross_entropy(logits, target)
loss.backward()

autograd_gradient = logits.grad.detach().clone()

with torch.no_grad():
    probabilities = torch.softmax(
        logits,
        dim=-1,
    )

    one_hot_target = torch.zeros_like(probabilities)

    one_hot_target[0, target.item()] = 1.0

    manual_gradient = probabilities - one_hot_target

print("probabilities:", probabilities)
print("autograd gradient:", autograd_gradient)
print("manual gradient:", manual_gradient)

assert torch.allclose(autograd_gradient, manual_gradient, atol=1e-6, rtol=1e-5)

print("Cross Entropy backward test passed.")

probabilities: tensor([[0.0854, 0.2321, 0.6308, 0.0518]])
autograd gradient: tensor([[ 0.0854, -0.7679,  0.6308,  0.0518]])
manual gradient: tensor([[ 0.0854, -0.7679,  0.6308,  0.0518]])
Cross Entropy backward test passed.


## 31. Mean Reduction 会改变 Gradient Scale

对于一个 Sample：

$$
L_n
$$

有：

$$
\nabla_zL_n
=
p_n-y_n
$$

但是实际训练时通常使用：

$$
L
=
\frac{1}{N}
\sum_{n=1}^{N}L_n
$$

因此：

$$
\frac{\partial L}
{\partial z_n}
=
\frac{1}{N}
(p_n-y_n)
$$

所以如果：

`reduction="mean"`

Batch 中每个 Sample 的 Logit Gradient 实际为：

$$
\boxed{
\frac{p-y}{N}
}
$$

如果使用：

`reduction="sum"`

则为：

$$
\boxed{
p-y
}
$$

没有：

$$
\frac{1}{N}
$$

这个因子。

这就是为什么不同 Reduction 会改变 Gradient Scale。


In [22]:
torch.manual_seed(42)

N = 4
V = 6

logits = torch.randn(N, V, requires_grad=True)
targets = torch.randint(low=0, high=V, size=(N,))

loss = F.cross_entropy(logits, targets, reduction="mean")
loss.backward()

autograd_gradient = logits.grad.detach().clone()

with torch.no_grad():
    probabilities = torch.softmax(logits, dim=-1)

    one_hot_targets = torch.zeros_like(probabilities)
    one_hot_targets.scatter_(dim=1, index=targets.unsqueeze(1), value=1.0)

    manual_gradient = (probabilities - one_hot_targets) / N

assert torch.allclose(autograd_gradient, manual_gradient, atol=1e-6, rtol=1e-5)

print("Mean-reduction backward test passed.")

Mean-reduction backward test passed.


## 32. Cross Entropy Reduction

Cross Entropy 通常可以有三种 Reduction。

### `reduction="none"`

保留每个 Sample 的 Loss：

$$
(N,V),(N)
\rightarrow
(N)
$$

例如：

$$
[0.4,1.2,0.8,2.1]
$$

---

### `reduction="sum"`

把所有 Sample Loss 相加：

$$
(N)
\rightarrow
()
$$

即：

$$
L
=
\sum_{n=1}^{N}L_n
$$

---

### `reduction="mean"`

取平均：

$$
L
=
\frac{1}{N}
\sum_{n=1}^{N}L_n
$$

这是训练中最常见的默认形式。

在 Language Model 中：

$$
N
$$

通常不是单纯的 Batch Size。

Flatten 以后：

$$
N=B\times T
$$

因此 Mean Cross Entropy 通常是在所有参与训练的 Token Position 上取平均。


In [23]:
torch.manual_seed(42)

logits = torch.randn(4, 10)
targets = torch.tensor([1, 3, 4, 7])

loss_none = F.cross_entropy(logits, targets, reduction="none")
loss_sum = F.cross_entropy(logits, targets, reduction="sum")
loss_mean = F.cross_entropy(logits, targets, reduction="mean")

print("none:", loss_none)
print("sum:", loss_sum)
print("mean:", loss_mean)

assert torch.allclose(loss_none.sum(), loss_sum)
assert torch.allclose(loss_none.mean(), loss_mean)


none: tensor([1.6468, 3.1059, 3.1586, 3.3712])
sum: tensor(11.2826)
mean: tensor(2.8206)


## 33. Next-Token Prediction

到目前为止，我们一直写：

$$
inputs.shape=(B,T)
$$

$$
targets.shape=(B,T)
$$

但 Language Model 有一个非常重要的细节：

> Position $t$ 的输入不是预测 Position $t$ 自己，而是预测下一个 Token。

假设文本经过 Tokenization 后：

$$
[x_0,x_1,x_2,x_3,x_4]
$$

Autoregressive Language Model 学习：

$$
P(x_1|x_0)
$$

$$
P(x_2|x_0,x_1)
$$

$$
P(x_3|x_0,x_1,x_2)
$$

$$
P(x_4|x_0,x_1,x_2,x_3)
$$

因此训练 Pair 应该是：

Input：

$$
[x_0,x_1,x_2,x_3]
$$

Target：

$$
[x_1,x_2,x_3,x_4]
$$

也就是 Target 相对于 Input：

向左 Shift 一个 Token。


## 36. Input / Target Shift

假设一句话 Token IDs：

$$
[10,23,51,7,9,4]
$$

那么：

Input：

$$
[10,23,51,7,9]
$$

Target：

$$
[23,51,7,9,4]
$$

对应关系：

| Input Position | 当前 Token | Target |
|---:|---:|---:|
| 0 | 10 | 23 |
| 1 | 23 | 51 |
| 2 | 51 | 7 |
| 3 | 7 | 9 |
| 4 | 9 | 4 |

模型在 Position 0 输出：

$$
logits[0]
$$

用于预测：

$$
23
$$

Position 1 输出：

$$
logits[1]
$$

用于预测：

$$
51
$$

依此类推。

因此 Language Modeling 本质上是在每一个 Sequence Position 上做一次：

Vocabulary Classification。


In [24]:
tokens = torch.tensor([10, 23, 51, 7, 9, 4])

inputs = tokens[:-1]
targets = tokens[1:]

print("tokens:", tokens)
print("inputs:", inputs)
print("targets:", targets)

tokens: tensor([10, 23, 51,  7,  9,  4])
inputs: tensor([10, 23, 51,  7,  9])
targets: tensor([23, 51,  7,  9,  4])


## 35. Batch 中的 Shift

如果原始 Token Batch：

$$
tokens.shape=(B,T+1)
$$

可以构造：

Input：

`tokens[:, :-1]`

Target：

`tokens[:, 1:]`

因此：

$$
inputs.shape=(B,T)
$$

$$
targets.shape=(B,T)
$$

例如：

$$
tokens.shape=(32,129)
$$

那么：

$$
inputs.shape=(32,128)
$$

$$
targets.shape=(32,128)
$$

模型输入：

$$
inputs
$$

输出：

$$
logits.shape=(32,128,V)
$$

然后与：

$$
targets.shape=(32,128)
$$

计算 Cross Entropy。


In [25]:
torch.manual_seed(42)

B = 3
T = 5
V = 100

tokens = torch.randint(low=0, high=V, size=(B, T + 1))

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

print("tokens:", tokens.shape)
print("inputs:", inputs.shape)
print("targets:", targets.shape)

assert inputs.shape == (B, T)
assert targets.shape == (B, T)


tokens: torch.Size([3, 6])
inputs: torch.Size([3, 5])
targets: torch.Size([3, 5])


## 36. Next-Token Prediction 与 Causal Attention

现在出现一个重要问题。

如果我们让 Position：

$$
t
$$

预测：

$$
x_{t+1}
$$

那么 Position：

$$
t
$$

在 Transformer 中绝对不能提前看到：

$$
x_{t+1}
$$

否则模型相当于：

> 直接看答案。

因此 Attention 必须满足：

Position $t$ 只能看到：

$$
0,1,\dots,t
$$

不能看到：

$$
t+1,t+2,\dots
$$

这就是：

Causal Mask。

例如 Sequence Length：

$$
T=4
$$

可见关系可以写成：

$$
\begin{bmatrix}
1 & 0 & 0 & 0\\
1 & 1 & 0 & 0\\
1 & 1 & 1 & 0\\
1 & 1 & 1 & 1
\end{bmatrix}
$$

因此：

Next-Token Target Shift

和：

Causal Attention

实际上是一套完整机制的两部分。

Target Shift 决定：

> 当前位置预测谁。

Causal Mask 决定：

> 当前位置允许看见谁。

后面学习 Attention 时会完整实现。


## 37. Mini Language Model Loss Pipeline

现在还没有正式实现 Transformer。

但我们已经可以模拟 Language Model 的最后几步：

$$
Token\ IDs
$$

$$
\downarrow
$$

$$
Embedding
$$

$$
\downarrow
$$

$$
Hidden\ States
$$

$$
\downarrow
$$

$$
LM\ Head
$$

$$
\downarrow
$$

$$
Logits
$$

$$
\downarrow
$$

$$
Cross\ Entropy
$$

$$
\downarrow
$$

$$
Loss
$$

为了只关注 Shape，我们暂时使用：

`nn.Embedding`

和：

`nn.Linear`

模拟模型。

之后自己实现完整 Transformer 时，这部分 Loss Pipeline 基本不需要改变。


In [27]:
import torch.nn as nn

torch.manual_seed(42)

B = 2
T = 5
V = 32
D = 16

tokens = torch.randint(low=0, high=V, size=(B, T + 1))

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

embedding = nn.Embedding(num_embeddings=V, embedding_dim=D)
lm_head = nn.Linear(in_features=D, out_features=V, bias=False)

hidden_states = embedding(inputs)

logits = lm_head(hidden_states)

loss = F.cross_entropy(
    logits.reshape(-1, V),
    targets.reshape(
        -1,
    ),
)

print("tokens:", tokens.shape)
print("inputs:", inputs.shape)
print("targets:", targets.shape)
print("hidden states:", hidden_states.shape)
print("logits:", logits.shape)
print("loss:", loss.shape)
print("loss value:", loss.item())

tokens: torch.Size([2, 6])
inputs: torch.Size([2, 5])
targets: torch.Size([2, 5])
hidden states: torch.Size([2, 5, 16])
logits: torch.Size([2, 5, 32])
loss: torch.Size([])
loss value: 3.577172040939331


## 38. 从 Loss 回到模型参数

最终：

$$
loss
$$

是 Scalar。

调用：

`loss.backward()`

以后，Autograd 根据 Chain Rule：

$$
\frac{\partial L}
{\partial \theta}
$$

计算所有可训练参数的 Gradient。

这里：

$$
\theta
$$

包括：

Embedding Weight：

$$
W_E
$$

以及 LM Head Weight：

$$
W_{LM}
$$

Gradient Flow：

$$
Loss
$$

$$
\downarrow
$$

$$
Logits
$$

$$
\downarrow
$$

$$
LM\ Head
$$

$$
\downarrow
$$

$$
Hidden\ States
$$

$$
\downarrow
$$

$$
Embedding
$$

而我们刚才已经知道：

Cross Entropy 首先产生：

$$
\frac{\partial L}
{\partial logits}
$$

其核心形式就是：

$$
p-y
$$

之后再通过 Linear、Transformer、Embedding 继续向后传播。

所以：

`loss.backward()`

虽然只有一行代码，

其背后实际上是一条完整的 Chain Rule。


In [28]:
embedding.zero_grad()
lm_head.zero_grad()

loss.backward()

print("embedding weight grad:", embedding.weight.grad.shape)
print("lm head weight grad:", lm_head.weight.grad.shape)

assert embedding.weight.grad is not None
assert lm_head.weight.grad is not None


embedding weight grad: torch.Size([32, 16])
lm head weight grad: torch.Size([32, 16])


## 39. Random Baseline：为什么是 $\log V$？

这是 Language Model Training 中一个非常有用的 Sanity Check。

假设模型完全没有学到东西。

对于 Vocabulary Size：

$$
V
$$

如果模型对每个 Token 都给出 Uniform Probability：

$$
p_i
=
\frac{1}{V}
$$

正确 Token 的 Probability：

$$
p_y
=
\frac{1}{V}
$$

Cross Entropy：

$$
L
=
-\log
\left(
\frac{1}{V}
\right)
$$

因此：

$$
\boxed{
L=\log V
}
$$

例如：

$$
V=10000
$$

那么：

$$
\log(10000)
\approx
9.21
$$

如果刚初始化的模型 Loss 大约在：

$$
\log V
$$

附近，通常是合理的。

如果一开始 Loss 是：

$$
100
$$

或者：

$$
NaN
$$

往往说明实现存在问题。

这个 Sanity Check 以后训练 TinyStories 时会非常有用。


In [29]:
import math

for vocab_size in [10, 100, 1_000, 10_000, 50_000]:
    baseline_loss = math.log(vocab_size)

    print(f"V={vocab_size:>6}", f"log(V)={baseline_loss:.4f}")


V=    10 log(V)=2.3026
V=   100 log(V)=4.6052
V=  1000 log(V)=6.9078
V= 10000 log(V)=9.2103
V= 50000 log(V)=10.8198


## 40. Uniform Prediction Test

如果所有 Logits 完全相同：

$$
z_1=z_2=\dots=z_V
$$

那么经过 Softmax：

$$
p_i
=
\frac{1}{V}
$$

因此 Cross Entropy 应该严格接近：

$$
\log(V)
$$

这可以作为 Cross Entropy 实现的另一个 Unit Test。


In [30]:
N = 32
V = 100

logits = torch.zeros(N, V)
targets = torch.randint(low=0, high=V, size=(N,))

loss = F.cross_entropy(logits, targets)

expected_loss = math.log(V)

print("loss:", loss.item())
print("log(V):", expected_loss)

assert abs(loss.item() - expected_loss) < 1e-6

print("Uniform baseline test passed.")

loss: 4.605170249938965
log(V): 4.605170185988092
Uniform baseline test passed.


## 41. 从 Cross Entropy 到 Perplexity

Language Modeling 中还经常看到一个指标：

Perplexity。

如果平均 Cross Entropy 为：

$$
L
$$

那么：

$$
\boxed{
PPL=e^L
}
$$

也就是：

$$
Perplexity
=
\exp(
CrossEntropy
)
$$

如果：

$$
L=\log(V)
$$

那么：

$$
PPL
=
e^{\log(V)}
=
V
$$

所以对于 Vocabulary 上完全 Uniform 的模型：

$$
PPL=V
$$

直觉上：

Perplexity 可以粗略理解成：

> 模型在每个位置上“像是在多少个候选 Token 中犹豫”。

例如：

$$
PPL=20
$$

比：

$$
PPL=100
$$

通常说明模型预测更加集中。

但需要注意：

不同 Tokenizer、Vocabulary、Dataset 上的 Perplexity 不应该简单横向比较。

因为 Tokenization 会直接改变：

- Sequence Length；
- Token Distribution；
- Cross Entropy；
- Perplexity。


In [31]:
loss_value = torch.tensor(3.0)

perplexity = torch.exp(loss_value)

print("cross entropy:", loss_value.item())
print("perplexity:", perplexity.item())

cross entropy: 3.0
perplexity: 20.08553695678711


## 42. Training vs Inference

这是非常重要的工程区别。

### Training

通常直接：

`logits -> cross_entropy`

而不是：

`logits -> softmax -> cross_entropy`

原因是 Cross Entropy 可以直接通过：

$$
LogSumExp
-
TargetLogit
$$

稳定计算。

所以：

`F.cross_entropy(...)`

输入应该是 Raw Logits。

---

### Inference

如果我们真的需要：

Probability Distribution

例如：

- Sampling；
- Top-k；
- Top-p；
- 查看 Token Probability；

才会显式计算：

$$
softmax(logits)
$$

因此：

Training：

$$
logits
\rightarrow
CrossEntropy
$$

Inference：

$$
logits
\rightarrow
Softmax
\rightarrow
Sampling
$$

不要混淆这两个场景。


## 43. Common Mistake：Double Softmax

错误写法：

`F.cross_entropy(torch.softmax(logits, dim=-1), targets)`

这是错误的。

因为：

`F.cross_entropy`

本身期望输入：

Raw Logits。

它内部对应的是稳定的：

$$
LogSoftmax
+
NLLLoss
$$

如果提前 Softmax：

$$
logits
\rightarrow
probabilities
$$

然后再让 Cross Entropy 把这些 Probability 当作 Logits，

数学意义就已经改变了。

正确写法：

`F.cross_entropy(logits, targets)`


In [32]:
torch.manual_seed(42)

logits = torch.randn(8, 20)
targets = torch.randint(low=0, high=20, size=(8,))

correct_loss = F.cross_entropy(logits, targets)
incorrect_loss = F.cross_entropy(torch.softmax(logits, dim=-1), targets)

print("correct:", correct_loss.item())
print("incorrect:", incorrect_loss.item())

correct: 3.730907917022705
incorrect: 3.017784833908081


## 44. Common Mistake：Target Dtype

普通 Token Classification 中：

Target 表示：

Vocabulary Index。

因此通常应该使用整数类型：

`torch.long`

例如：

$$
targets=
[3,17,8,2]
$$

正确：

`dtype=torch.long`

而不是：

`float32`

这是因为 Target 不是 Probability。

它是：

Class Index。

这和 Embedding 的 Input Token ID 一样：

Token ID 本质上是离散整数索引。


In [33]:
targets = torch.tensor([1, 3, 5], dtype=torch.long)

print(targets.dtype)

assert targets.dtype == torch.long


torch.int64


## 45. Common Mistake：Wrong Dimension

我们自己的 Language Model Logits 通常使用：

$$
(B,T,V)
$$

也就是 Vocabulary Dimension 在最后。

因此自己实现 Softmax 时通常：

`dim=-1`

但是 PyTorch 某些 Loss API 的一般 Classification 接口习惯是：

$$
(N,C,\dots)
$$

也就是 Class Dimension 在：

$$
dim=1
$$

所以实际工程中必须先确认 API 的 Shape Contract。

对于本课程自己的实现，我们会明确保持：

$$
(B,T,V)
$$

然后 Flatten：

$$
(BT,V)
$$

再计算：

Cross Entropy。

这样可以避免 Dimension Ambiguity。


## 46. Common Mistake：Input 和 Target 没有 Shift

错误：

Input：

$$
[x_0,x_1,x_2,x_3]
$$

Target：

$$
[x_0,x_1,x_2,x_3]
$$

这会让模型学习：

> 在看到当前 Token 后预测当前 Token。

对于具有 Residual Connection 的 Transformer，这不是我们需要的 Autoregressive Language Modeling Objective。

正确：

Input：

$$
[x_0,x_1,x_2,x_3]
$$

Target：

$$
[x_1,x_2,x_3,x_4]
$$

也就是：

$$
target_t
=
input_{t+1}
$$

训练语言模型时：

Target Shift

必须和：

Causal Mask

一起正确实现。


## 47. `reshape` 与 `view`

在 Language Model Cross Entropy 中经常需要：

$$
(B,T,V)
\rightarrow
(BT,V)
$$

一种写法：

`logits.view(-1, V)`

另一种写法：

`logits.reshape(-1, V)`

对于初学阶段，我更推荐：

`reshape`

原因是 Tensor 经过某些：

- transpose；
- permute；

操作以后可能不是 Contiguous。

此时：

`view`

可能失败。

而：

`reshape`

会在必要时处理底层布局问题。

后面学习 Tensor Memory Layout 时，我们会详细讨论：

- contiguous；
- stride；
- view；
- reshape；
- transpose。

目前只需要记住：

在不确定 Contiguity 时：

`reshape`

更稳妥。


## 48. Scalar Loss

如果：

`reduction="none"`

Cross Entropy 返回：

$$
(N)
$$

它不是 Scalar。

直接：

`loss.backward()`

通常需要显式提供上游 Gradient。

训练中更加常见的是：

$$
(N)
\rightarrow
mean
\rightarrow
()
$$

然后：

`loss.backward()`

因此典型训练流程：

`loss = losses.mean()`

然后：

`loss.backward()`

后面学习 Training Loop 时会反复使用这个模式。


## 49. Cross Entropy with Reduction

现在把自己的 Cross Entropy 稍微扩展一下。

希望支持：

- `none`
- `sum`
- `mean`

核心仍然是：

$$
loss_i
=
LogSumExp(z_i)
-
z_{i,y_i}
$$

区别只在最后的 Reduction。


In [34]:
def cross_entropy_from_scratch(
    logits: torch.Tensor, targets: torch.Tensor, reduction: str = "mean"
) -> torch.Tensor:
    if logits.ndim != 2:
        raise ValueError("logits must have shape (N, V)")

    if targets.ndim != 1:
        raise ValueError("targets must have shape (N,)")

    if logits.shape[0] != targets.shape[0]:
        raise ValueError("logits and targets must have the same N")

    log_normalizer = torch.logsumexp(logits, dim=-1)

    row_indices = torch.arange(logits.shape[0], device=logits.device)
    target_logits = logits[row_indices, targets]

    losses = log_normalizer - target_logits

    if reduction == "none":
        return losses

    if reduction == "sum":
        return losses.sum()

    if reduction == "mean":
        return losses.mean()

    raise ValueError(f"Unsupported reduction: {reduction}")


In [35]:
torch.manual_seed(42)

N = 32
V = 128

logits = torch.randn(N, V)
targets = torch.randint(low=0, high=V, size=(N,), dtype=torch.long)

for reduction in ["none", "sum", "mean"]:
    our_loss = cross_entropy_from_scratch(logits, targets, reduction=reduction)
    reference_loss = F.cross_entropy(logits, targets, reduction=reduction)

    assert torch.allclose(our_loss, reference_loss, atol=1e-6, rtol=1e-5)

    print(f"{reduction}: passed")


none: passed
sum: passed
mean: passed


## 50. 不只测试 Forward，也测试 Backward

对于 Neural Network Primitive：

只验证 Forward 不够。

因为即使：

$$
forward
$$

输出正确，

如果 Autograd Graph 中某一步写错：

Backward 仍然可能存在问题。

因此一个比较完整的 Primitive Test 应包含：

1. Shape Test；
2. Forward Numerical Test；
3. Edge Case Test；
4. Backward Gradient Test。

对于我们的 Cross Entropy：

PyTorch Tensor Operation 本身支持 Autograd，

所以只要我们的数学表达式正确，

Gradient 应该能够自动正确传播。

现在和：

`F.cross_entropy`

比较 Logits Gradient。


In [36]:
torch.manual_seed(42)

N = 8
V = 32

logits_ours = torch.randn(N, V, requires_grad=True)
logits_reference = logits_ours.detach().clone().requires_grad_(True)

targets = torch.randint(low=0, high=V, size=(N,))

our_loss = cross_entropy_from_scratch(logits_ours, targets)
reference_loss = F.cross_entropy(logits_reference, targets)

our_loss.backward()
reference_loss.backward()

assert torch.allclose(our_loss, reference_loss, atol=1e-6, rtol=1e-5)
assert torch.allclose(logits_ours.grad, logits_reference.grad, atol=1e-6, rtol=1e-5)

print("Forward and backward tests passed.")


Forward and backward tests passed.


## 51. 一个 Primitive 应该怎样测试？

现在把之前零散的测试组合起来。

我们至少测试：

### Shape

`reduction="none"`

应该得到：

$$
(N)
$$

### Forward

和：

`F.cross_entropy`

对齐。

### Backward

Gradient 和 Reference 对齐。

### Large Logits

测试 Numerical Stability。

### Uniform Logits

验证：

$$
L=\log(V)
$$

这已经非常接近 Assignment 中真正的 Unit Test 思维：

> 不是“运行没有报错”就算正确。

而是主动定义：

Correctness Contract。


In [37]:
def test_cross_entropy_from_scratch() -> None:
    torch.manual_seed(42)

    N = 16
    V = 64

    logits = torch.randn(N, V)
    targets = torch.randint(low=0, high=V, size=(N,))

    # Shape test
    losses = cross_entropy_from_scratch(logits, targets, reduction="none")

    assert losses.shape == (N,)

    # Forward test
    our_loss = cross_entropy_from_scratch(logits, targets, reduction="mean")
    reference_loss = F.cross_entropy(logits, targets, reduction="mean")

    assert torch.allclose(our_loss, reference_loss, atol=1e-6, rtol=1e-5)

    # Large-logit stability test
    large_logits = torch.randn(N, V) * 1000.0

    large_loss = cross_entropy_from_scratch(large_logits, targets)
    reference_large_loss = F.cross_entropy(large_logits, targets)

    assert torch.isfinite(large_loss)
    assert torch.allclose(large_loss, reference_large_loss, atol=1e-4, rtol=1e-5)

    # Uniform baseline test
    uniform_logits = torch.zeros(N, V)
    uniform_loss = cross_entropy_from_scratch(uniform_logits, targets)

    expected = torch.log(torch.tensor(float(V)))

    assert torch.allclose(uniform_loss, expected, atol=1e-6, rtol=1e-5)

    # Backward test
    logits_ours = torch.randn(N, V, requires_grad=True)

    logits_reference = logits_ours.detach().clone().requires_grad_(True)

    our_loss = cross_entropy_from_scratch(logits_ours, targets)
    reference_loss = F.cross_entropy(logits_reference, targets)

    our_loss.backward()
    reference_loss.backward()

    assert torch.allclose(logits_ours.grad, logits_reference.grad, atol=1e-6, rtol=1e-5)


test_cross_entropy_from_scratch()

print("All Cross Entropy tests passed.")

All Cross Entropy tests passed.


## 52. 当模型已经非常自信时 Gradient 会怎样？

我们知道：

$$
\nabla_zL
=
p-y
$$

假设正确类别是：

$$
y
$$

如果模型已经非常正确：

$$
p_y\approx1
$$

那么：

正确类别 Gradient：

$$
p_y-1
\approx
0
$$

其他类别：

$$
p_i\approx0
$$

所以：

$$
p_i-0
\approx0
$$

因此：

$$
\nabla_zL
\approx0
$$

也就是说：

> 当模型已经非常正确时，Cross Entropy 不会继续产生很大的更新。

反过来，如果模型对错误类别极其自信：

例如正确类别 Probability：

$$
p_y\approx0
$$

那么正确类别 Gradient：

$$
p_y-1
\approx
-1
$$

产生很强的修正信号。

这也是 Cross Entropy 非常适合 Classification 的原因之一。


## 53. Loss 与 Accuracy 的区别

假设正确 Token 是：

$$
A
$$

模型 1：

$$
P(A)=0.51
$$

其他最大 Probability：

$$
0.49
$$

模型预测正确。

模型 2：

$$
P(A)=0.99
$$

模型也预测正确。

从 Accuracy 看：

两个都是：

$$
100\%
$$

但 Cross Entropy：

模型 1：

$$
-\log(0.51)
$$

约为：

$$
0.673
$$

模型 2：

$$
-\log(0.99)
$$

约为：

$$
0.010
$$

因此 Cross Entropy 不只是关心：

> 最大 Probability 对不对。

还关心：

> 模型有多相信正确答案。

这使得它能够提供连续、可微的 Training Signal。


# From PyTorch Fundamentals to Language Modeling

目前已经掌握的知识开始真正连接起来。

### Tensor

理解：

$$
shape
$$

$$
dtype
$$

$$
device
$$

---

### Broadcasting

理解：

$$
(B,T,V)
/
(B,T,1)
$$

为什么成立。

Stable Softmax 中已经实际使用。

---

### Matrix Multiplication

理解：

$$
XW
$$

以及：

$$
(B,T,D)
\times
(D,V)
\rightarrow
(B,T,V)
$$

这就是 LM Head。

---

### `einsum`

后面 Attention 会大量使用。

---

### Autograd

理解：

`loss.backward()`

如何沿 Computation Graph 传播 Gradient。

---

### `nn.Module`

后面所有 Transformer Component 都会继承它。

---

### Linear

实现：

$$
XW^T+b
$$

---

### Embedding

实现：

$$
Token\ ID
\rightarrow
Vector
$$

---

### Softmax

实现：

$$
Logits
\rightarrow
Probability
$$

---

### Cross Entropy

实现：

$$
Prediction
+
Target
\rightarrow
Loss
$$

因此现在已经完成了：

$$
Input
$$

和：

$$
Training\ Objective
$$

两端。

接下来要开始真正构造中间的：

Neural Network。
